# Exercise 4: Model Comparison -- nano vs mini vs 4.1

Sends the same real-world troubleshooting prompt to `gpt-4.1-nano`, `gpt-4.1-mini`,  
and `gpt-4.1`, then compares latency, token usage, and estimated cost.  
This is the kind of benchmark you'd run before choosing a model tier for production.

In [1]:
import time

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

## Define the prompt and pricing

In [2]:
PROMPT = (
    "A customer says: 'We're seeing 3x higher latency on our RAG pipeline since "
    "migrating to the new embedding model. Our p99 went from 200ms to 600ms. "
    "What should we investigate?' "
    "Give a structured troubleshooting plan."
)

MODELS = ["gpt-4.1-nano", "gpt-4.1-mini", "gpt-4.1"]

# Approximate pricing per 1M tokens
PRICING = {
    "gpt-4.1-nano":  {"input": 0.10, "output": 0.40},
    "gpt-4.1-mini":  {"input": 0.40, "output": 1.60},
    "gpt-4.1":       {"input": 2.00, "output": 8.00},
}

## Run the same prompt against all three models

In [3]:
results = []

for model in MODELS:
    print(f"\n{'='*60}")
    print(f"MODEL: {model}")
    print(f"{'='*60}")

    start = time.time()
    response = client.responses.create(model=model, input=PROMPT)
    elapsed = time.time() - start

    cost_input = (response.usage.input_tokens / 1_000_000) * PRICING[model]["input"]
    cost_output = (response.usage.output_tokens / 1_000_000) * PRICING[model]["output"]
    cost_total = cost_input + cost_output

    results.append({
        "model": model,
        "elapsed": elapsed,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "cost": cost_total,
        "text": response.output_text,
    })

    print(f"\n{response.output_text}")
    print(f"\n--- {model} stats ---")
    print(f"Latency:  {elapsed:.2f}s")
    print(f"Tokens:   {response.usage.input_tokens} in, {response.usage.output_tokens} out")
    print(f"Est cost: ${cost_total:.6f}")


MODEL: gpt-4.1-nano

Certainly! Here's a structured troubleshooting plan to investigate the increased latency in your RAG (Retrieval-Augmented Generation) pipeline after migrating to the new embedding model:

---

### 1. **Confirm the Issue Scope and Details**
- **Verify the Metrics:**
  - Confirm the p99 latency change (200ms → 600ms).
  - Check if other latency metrics (average, p95, p99.9) are also affected.
- **Identify Affected Components:**
  - Embedding generation
  - Similarity search / retrieval
  - Post-processing / filtering
  - Model inference (if involved after retrieval)

### 2. **Baseline and Compare**
- **Compare with previous setup:**
  - Isolate the components that changed (the embedding model) versus other pipeline parts.
- **Data Volume & Patterns:**
  - Has the document corpus or query load increased?
  
---

### 3. **Investigate Embedding Model Inference**
- **Model Complexity & Size:**
  - Is the new embedding model larger or more complex?
  - Check parameters, 

## Comparison summary table

In [4]:
print(f"{'Model':<18} {'Latency':>8} {'In tok':>8} {'Out tok':>8} {'Cost':>12}")
print("-" * 60)
for r in results:
    print(f"{r['model']:<18} {r['elapsed']:>7.2f}s {r['input_tokens']:>8} {r['output_tokens']:>8} ${r['cost']:>10.6f}")

Model               Latency   In tok  Out tok         Cost
------------------------------------------------------------
gpt-4.1-nano          7.60s       56      745 $  0.000304
gpt-4.1-mini         12.79s       56      901 $  0.001464
gpt-4.1              11.50s       56     1007 $  0.008168


## Relative cost and latency vs gpt-4.1

In [5]:
print("--- Relative to gpt-4.1 ---")
base = results[2]  # gpt-4.1
for r in results:
    cost_ratio = r["cost"] / base["cost"] if base["cost"] > 0 else 0
    speed_ratio = r["elapsed"] / base["elapsed"] if base["elapsed"] > 0 else 0
    print(f"{r['model']:<18} {cost_ratio:>5.1%} the cost, {speed_ratio:>5.1%} the latency")

--- Relative to gpt-4.1 ---
gpt-4.1-nano        3.7% the cost, 66.1% the latency
gpt-4.1-mini       17.9% the cost, 111.2% the latency
gpt-4.1            100.0% the cost, 100.0% the latency


## Key tradeoff insight

In [6]:
print("nano:  Fastest & cheapest. Good for classification, routing, simple extraction.")
print("mini:  Sweet spot. Good quality at 5x less cost than full 4.1.")
print("4.1:   Best quality. Use for complex reasoning, nuanced customer-facing content.")

nano:  Fastest & cheapest. Good for classification, routing, simple extraction.
mini:  Sweet spot. Good quality at 5x less cost than full 4.1.
4.1:   Best quality. Use for complex reasoning, nuanced customer-facing content.


## Interview one-liner

> **Model selection is a cost-quality-latency tradeoff**: `nano` is ideal for high-volume routing/classification, `mini` is the production sweet spot for most tasks, and full `4.1` is reserved for complex reasoning where quality justifies the 20x cost premium.